In [1]:
import time

import torch
from torch import nn
from torch.nn import functional as F

import numpy as np

import tqdm as tqdm

import tiktoken

from model import ModelConfig, MyGPT
from dataloader import DataLoader

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


# Define tokens  and their encode/decode

In [3]:
# TokenizerName = 'Char'
TokenizerName = 'Tiktoken'

if TokenizerName=='Char':

    # TODO

    # set(text): find unique characteres
    # list(set(text)): convert to list
    # sorted(list(set(text))): sort
    Tokens = sorted(list(set(text)))
    NToken = len(Tokens)

    print('Number of tokens:', NToken)

    # encode: encode(list of character) = list of integer, index from Tokens list
    # decode: decode(list of integer) = list of chracter

    dict_c_to_i = {c:i_c for i_c, c in enumerate(Tokens)}
    dict_i_to_c = {i_c:c for i_c, c in enumerate(Tokens)}

    encode = lambda list_c: [dict_c_to_i[c] for c in list_c]
    decode = lambda list_i: ''.join([dict_i_to_c[i] for i in list_i])
    

if TokenizerName=='Tiktoken':
    
    enc = tiktoken.get_encoding("gpt2")

    NToken = enc.n_vocab

    encode = lambda list_c: enc.encode(list_c)
    decode = lambda list_i: enc.decode(list_i)


In [4]:
print('Number of Tokens:',NToken)
print(encode('My name is Jaesung'))
print(decode(encode('My name is Jaesung')))

Number of Tokens: 50257
[3666, 1438, 318, 13790, 274, 2150]
My name is Jaesung


# Now declare our model

In [5]:
_config = ModelConfig()

_config.DToken = NToken

_config.ContextSize = 1024

_config.DHead = 64

_config.NHeadQ = 6
_config.NHeadKV = 6

_config.NLayer = 6


In [6]:
my_model = MyGPT(_config).to(device)

my_model = torch.compile(my_model)

In [7]:
# print the number of parameters in the model
print(sum(p.numel() for p in my_model.parameters()), 'parameters')

49224960 parameters


# Generate with the initial model

In [8]:
input_text = 'Dog'

tmp_start_text = encode(input_text)

# generate from the model
print('Input text:')
print(input_text)

output_text = decode(my_model.generate(tmp_start_text, max_tokens=10)[0].tolist())
print('Output text:')
print(output_text)

Input text:
Dog
Output text:
Dogmedical Disaster ORDER predictably barrier Floracles understandably physics Medicaid


# Prepare training

In [9]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(my_model.parameters(), lr=3E-4)

In [ ]:
# Size of the batch for each training
NBatch = 4

# Max iteration of the training
NMaxIteration = 1000

# Logging
LogEvery = 50

# Evaluate the training every
EvalEvery = 100
# Number of iteration for every evaluation step
EvalIteration = 100

# Learning rate
LearningRate = 3E-4

# Prepare data

In [11]:
InputDataType = 'shakespeare'
InputDataType = 'openwebtext'


data_dir = f'data/{InputDataType}'

map_dls = dict()
for split in ['train', 'val']:
    map_dls[split] = DataLoader(NBatch, _config.ContextSize, data_dir, split)


In [12]:
# Function decorator
@torch.no_grad()
def estimate_loss():
    out = {}
    my_model.eval()
    for split in ['train', 'val']:
        # Make tensor placeholder
        losses = torch.zeros(EvalIteration)
        for k in range(EvalIteration):
            X, Y = map_dls[split].next_batch()
            X, Y = X.to(device), Y.to(device)
            loss = my_model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    my_model.train()
    return out

In [13]:
GlobalStepCounter = 0

In [14]:
for step in range(NMaxIteration):

    last_step = (step == NMaxIteration - 1)

    if step % EvalEvery == 0 or step == NMaxIteration - 1:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        if step > 0 and (step % 500 == 0 or last_step):
            # optionally write model checkpoints
            checkpoint_path = f'logs/model_{GlobalStepCounter:05d}.pt'
            checkpoint = {
                'model': my_model.state_dict(),
                'config': my_model.cfg,
                'step': step,
                'val_loss': losses['val']
            }
            # you might also want to add optimizer.state_dict() and
            # rng seeds etc., if you wanted to more exactly resume training
            torch.save(checkpoint, checkpoint_path)

    t0 = time.time()
    

    # sample a batch of data
    xb, yb = map_dls['train'].next_batch()
    xb, yb = xb.to(device), yb.to(device)

    # evaluate the loss
    loss = my_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    t1 = time.time()
    dt = t1 - t0 # time difference in seconds

    tokens_processed = NBatch * _config.ContextSize
    tokens_per_sec = tokens_processed / dt

    if step % LogEvery == 0 or step == NMaxIteration - 1:
        print(f"step {step:5d} | loss: {loss.item():.6f} dt: {dt*1000:.2f}ms | tok/sec: {tokens_per_sec:.2f}")

    GlobalStepCounter += 1

    if device=='mps':
        torch.mps.synchronize()

step 0: train loss 10.9919, val loss 10.9894
step     0 | loss: 10.998013 dt: 484.53ms | tok/sec: 8453.58
step    50 | loss: 7.965373 dt: 44.06ms | tok/sec: 92968.68
step 100: train loss 7.2958, val loss 6.9541
step   100 | loss: 7.240338 dt: 45.62ms | tok/sec: 89777.28
step   150 | loss: 7.485085 dt: 43.95ms | tok/sec: 93197.15
step 200: train loss 7.1096, val loss 6.7676
step   200 | loss: 6.793293 dt: 58.05ms | tok/sec: 70561.41
step   250 | loss: 6.739651 dt: 44.83ms | tok/sec: 91363.33
step 300: train loss 6.7895, val loss 7.1081
step   300 | loss: 7.004932 dt: 45.97ms | tok/sec: 89093.80
step   350 | loss: 6.721478 dt: 50.90ms | tok/sec: 80468.15
step 400: train loss 6.5521, val loss 6.6818
step   400 | loss: 6.587343 dt: 46.76ms | tok/sec: 87601.44
step   450 | loss: 6.887451 dt: 47.46ms | tok/sec: 86304.11
step 500: train loss 6.4965, val loss 6.5255
step   500 | loss: 6.550645 dt: 50.16ms | tok/sec: 81665.01
step   550 | loss: 6.766370 dt: 47.40ms | tok/sec: 86415.24
step 600:

In [15]:
map_dls['train'].NFullLoop

0

In [16]:
input_text = 'Hello'

tmp_start_text = encode(input_text)

# generate from the model
print('Input text:')
print(input_text)

output_text = decode(my_model.generate(tmp_start_text, max_tokens=10)[0].tolist())
print('Output text:')
print(output_text)


Input text:
Hello
Output text:
Hello and did not use the flight with a game did
